# Integrating neural networks with ODEs: Quarantine strength diagnosis in Italy during March - June 2020

## Introduction
In this tutorial, we will construct a Universal Differential Equation (UDE) based on the SIR model of epidemiology, with the aim of capturing quarantine effects in the region under consideration. We will show that the resulting UDE is not only good at estimation, but can be used as a powerful diagnostic tool. Using this tool, we can understand the real-time, data driven quarantine variation for any region. In this example, we choose Italy.

## Procedure
- First, we need to decide how to construct the UDE model. We will start with the SIR model, which has no information regarding quarantine effects. We will integrate a neural network in the SIR model to capture these effects. Intuitively, since quarantined (or isolated) people don't come into contact with the infected people, we need to subtract a term from the dI/dt equation in the SIR model. Thus, we inject a neural network as a subtraction term in the infected compartment, and the people it removes go to a quarantined compartment, T.
- Second, we will train the resulting UDE on the infected and recovered case counts published by the Johns Hopkins University repository.
- Third, after the model is trained, we will plot the quarantine strength the network recovered over time and see how it relates to the situation on the ground in Italy.

## Libraries

In [ ]:
Pollis.packages("PollisDatasets", "Lux", "OrdinaryDiffEq", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "OptimizationOptimJL", "ComponentArrays", "Mooncake", "Plots")

In [ ]:
using PollisDatasets, Lux, OrdinaryDiffEq, SciMLSensitivity, Optimization, OptimizationOptimisers, OptimizationOptimJL, ComponentArrays, Mooncake
using Random, Plots

## Data import and cleanup

The data consists of the cumulative infected, recovered and dead case counts in Italy from the detection of the 500th infected case (which we use as t = 0), 27 February 2020, to 31 May 2020, from the Johns Hopkins University CSSE repository. It is the `italycovid` dataset of PollisDatasets.

In [ ]:
d = dataset("italycovid")

Infected = Float64.(d.cases)
Recovered = Float64.(d.recovered)
Dead = Float64.(d.dead)

# The active cases: those infected who have neither recovered nor died
Infected = Infected .- Recovered .- Dead
datasize = length(Infected)
println(datasize, " days of data")

## Defining the UDE model
In this step, we define the beta, gamma and delta parameters of the model and the neural network that captures the quarantine strength, a function of S, I and R. We also define the UDE model as described in the procedure section at the beginning of this tutorial.

In [ ]:
rng = Xoshiro(50)
ann = Chain(Dense(3 => 10, relu), Dense(10 => 1))
p1, st1 = f64(Lux.setup(rng, ann))

parameter_array = [0.15, 0.013, 0.01]  # beta, gamma and delta

p0_vec = ComponentArray(layer_1 = p1, layer_2 = parameter_array)

u0 = [60000000.0, 593, 62, 10]  # S, I, R and T at t = 0
tspan = (0.0, Float64(datasize - 1))
t = 0.0:1.0:(datasize - 1)

# The network sees S, I and R in millions, hundreds of thousands and hundreds of thousands
quarantine(u, p) = abs(first(first(ann([u[1] / 1e7; u[2] / 1e5; u[3] / 1e5], p.layer_1, st1)))) * u0[1] / 10

function QSIR(u, p, t)
    beta, gamma, delta = abs.(p.layer_2)
    UDE_term = quarantine(u, p)
    return [
        -beta * u[1] * u[2] / u0[1],
        beta * u[1] * u[2] / u0[1] - gamma * u[2] - UDE_term * u[2] / u0[1],
        gamma * u[2] + delta * u[4],
        UDE_term * u[2] / u0[1] - delta * u[4],
    ]
end

prob = ODEProblem(QSIR, u0, tspan)

## Defining the loss function and performing the UDE training
The loss compares the logarithms of the counts, so that the early days, with small counts, count as much as the later ones. We use the Adam optimizer, followed by BFGS, for optimizing the neural network parameters.

In [ ]:
predict_adjoint(theta) = Array(solve(prob, Tsit5(); p = theta, saveat = t))

function loss_adjoint(theta, _)
    prediction = predict_adjoint(theta)
    return sum(abs2, log.(abs.(Infected)) .- log.(abs.(prediction[2, :] .+ prediction[4, :]))) +
           sum(abs2, log.(abs.(Recovered .+ Dead)) .- log.(abs.(prediction[3, :])))
end

losses = Float64[]
callback = function (state, l)
    push!(losses, l)
    length(losses) % 100 == 0 && println("Loss after ", length(losses), " iterations: ", l)
    return false
end

optf = OptimizationFunction(loss_adjoint, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, p0_vec)
res1 = solve(optprob, Adam(0.01); callback = callback, maxiters = 1000)

optprob2 = remake(optprob; u0 = res1.u)
res2 = solve(optprob2, Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 100)
println("Final loss: ", losses[end])

data_pred = predict_adjoint(res2.u)
p3n = res2.u
println("beta, gamma, delta: ", round.(abs.(p3n.layer_2); sigdigits = 3))

## Obtain Q(t): the quarantine strength recovered by the neural network

In [ ]:
Q_parameter = [quarantine(u, p3n) for u in eachcol(data_pred)]

## Plots

1. Data prediction

In [ ]:
bar(t, Infected; alpha = 0.5, label = "Data: Infected", color = :red)
plot!(t, data_pred[2, :] .+ data_pred[4, :]; xaxis = "Days post 500 infected", label = "Prediction", legend = :topleft, framestyle = :box, grid = :off, color = :red, linewidth = 4)
bar!(t, Recovered .+ Dead; alpha = 0.5, label = "Data: Recovered", color = :blue)
plot!(t, data_pred[3, :]; ylims = (-0.05 * maximum(Recovered .+ Dead), 1.5 * maximum(Recovered .+ Dead)), label = "Prediction ", color = :blue, linewidth = 4)

2. Q(t) compared with when lockdown was imposed in Italy, on day 11.

In [ ]:
D1 = diff(Q_parameter)
D2 = diff(D1)
Transition = findfirst(x -> x < 0, D2)

scatter(t, Q_parameter ./ u0[1]; xlims = (0, datasize + 1), xlabel = "Days post 500 infected", ylabel = "Q(t)", label = "Quarantine strength", color = :black, framestyle = :box, grid = :off, legend = :topleft)
plot!([11 - 0.01, 11 + 0.01], [0.0, 0.6 * maximum(Q_parameter ./ u0[1])]; lw = 3, color = :green, label = "Government Lockdown imposed", linestyle = :dash)
Transition === nothing || plot!([Transition - 0.01, Transition + 0.01], [0.0, 0.6 * maximum(Q_parameter ./ u0[1])]; lw = 3, color = :red, label = "Inflection point in learnt Q(t)", linestyle = :dash)
plot!()

## Discussion

The time at which an inflection point is seen in the plot of Q(t) can be compared with the time at which lockdown was imposed in Italy. In the original study, the two matched quite closely: an indication that the model can be used as a diagnostic tool. The study ran this code for 75 different countries.

This shows that integrating neural networks with ODE models not only leads to expressive models, but also to models which can be highly interpretable. Such neural network assisted ODE models can capture a wide range of complex effects which are difficult to capture with traditional compartment models, or which become too opaque with agent based models. Some of these effects are vaccination strength, the effects of early reopening, and virtual virus spread through Bluetooth tokens.